In [0]:
delivery_risk_features = spark.read.table("revenue_operations.gold.delivery_risk_features")

# Dropping timestamp columns (after feature engineering in next cell)
delivery_risk_features = delivery_risk_features.drop("order_purchase_timestamp", "estimated_delivery_date")
# Note: earliest_shipping_limit_date and order_approved_at will be dropped after feature engineering

# Dropping customer_city
delivery_risk_features = delivery_risk_features.drop("customer_city")
delivery_risk_features.printSchema()

In [0]:
from pyspark.sql import functions as F

# Engineer time-based features from earliest_shipping_limit_date
print("Engineering time-based features from shipping limit date...")

delivery_risk_features = delivery_risk_features.withColumn(
    "days_to_shipping_limit",
    F.datediff(F.col("earliest_shipping_limit_date"), F.col("order_approved_at"))
)

# Tight deadline flag (2 days or less)
delivery_risk_features = delivery_risk_features.withColumn(
    "tight_deadline_flag",
    F.when(F.col("days_to_shipping_limit") <= 2, 1).otherwise(0)
)

# Day of week of shipping limit (1=Sunday, 7=Saturday)
delivery_risk_features = delivery_risk_features.withColumn(
    "shipping_limit_day_of_week",
    F.dayofweek(F.col("earliest_shipping_limit_date"))
)

# Weekend flag for shipping limit (1=weekend, 0=weekday)
delivery_risk_features = delivery_risk_features.withColumn(
    "shipping_limit_weekend_flag",
    F.when(F.dayofweek(F.col("earliest_shipping_limit_date")).isin([1, 7]), 1).otherwise(0)
)

print("Created 4 new time-based features:")
print("  - days_to_shipping_limit: Days between approval and shipping deadline")
print("  - tight_deadline_flag: 1 if deadline ≤ 2 days, else 0")
print("  - shipping_limit_day_of_week: 1-7 (1=Sunday)")
print("  - shipping_limit_weekend_flag: 1 if weekend deadline, else 0")

# Display sample
delivery_risk_features.select(
    "order_approved_at",
    "earliest_shipping_limit_date",
    "days_to_shipping_limit",
    "tight_deadline_flag",
    "shipping_limit_day_of_week",
    "shipping_limit_weekend_flag"
).display()

### Time based splitting data into training, validation and test sets.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Extract year and month from 'order_approved_at'

date_parsed = delivery_risk_features.withColumn(
    "order_approved_year_month", 
    F.date_format(F.col("order_approved_at"), "yyyy-MM")
)

year_month_counts = date_parsed.groupBy("order_approved_year_month").agg(
    F.count("*").alias("row_count")
).orderBy(F.col("order_approved_year_month").asc())

# Add cumulative sum of row counts
year_month_counts = year_month_counts.withColumn(
    "cum_row_count",
    F.sum("row_count").over(Window.orderBy("order_approved_year_month"))
)

total_rows = year_month_counts.agg(F.sum("row_count").alias("total")).collect()[0]["total"]

year_month_counts = year_month_counts.withColumn(
    "percent_of_total",
    (F.col("cum_row_count") / F.lit(total_rows)) * 100
)

display(year_month_counts)

Time based splitting for training, validations and testing plus data separated for model drift detection will be considered as future data, the splits planned by order_approved_at dates will be as follows:

- training = 2016-9 to 2018-3 that is roughly 66.5% of the rows
- validation = 2018-4 to 2018-5 that is roughly 14% of the rows
- test = 2018-6 to 2018-7 that is roughly 13% of the rows
- Future data = 2018-8 that is roughly 7% of the rows

In [0]:
# Training data
train = delivery_risk_features.filter((F.col("order_approved_at") < "2018-04-01") & (F.col("order_approved_at") >= "2016-08-01"))

# Validation 
val = delivery_risk_features.filter((F.col("order_approved_at") >= "2018-04-01") & (F.col("order_approved_at") < "2018-06-01"))

# Testing
test = delivery_risk_features.filter((F.col("order_approved_at") >= "2018-06-01") & (F.col("order_approved_at") < "2018-08-01"))

# Future data
future_data = delivery_risk_features.filter((F.col("order_approved_at") >= "2018-08-01") & (F.col("order_approved_at") < "2018-09-01")) 

In [0]:
# Drop timestamp columns from each split now that we've used them for splitting
train = train.drop("order_approved_at", "earliest_shipping_limit_date")
val = val.drop("order_approved_at", "earliest_shipping_limit_date")
test = test.drop("order_approved_at", "earliest_shipping_limit_date")
future_data = future_data.drop("order_approved_at", "earliest_shipping_limit_date")

print("✓ Dropped timestamp columns from all splits")
print("  - order_approved_at")
print("  - earliest_shipping_limit_date")
print("\nTime-based features retained:")
print("  - days_to_shipping_limit")
print("  - tight_deadline_flag")
print("  - shipping_limit_day_of_week")
print("  - shipping_limit_weekend_flag")

In [0]:
# Count rows in each split
train_count = train.count()
val_count = val.count()
test_count = test.count()
future_count = future_data.count()
original_count = delivery_risk_features.count()

# Calculate total from splits
split_total = train_count + val_count + test_count + future_count

# Display results
print(f"Training:   {train_count:,}")
print(f"Validation: {val_count:,}")
print(f"Test:       {test_count:,}")
print(f"Future:     {future_count:,}")
print(f"─" * 25)
print(f"Split Total:    {split_total:,}")
print(f"Original Total: {original_count:,}")
print(f"Difference:     {split_total - original_count:,}")

### Preprocessing Data only for training set

In [0]:
# Preprocessing the data
from pyspark.sql.functions import column 

# Checking for null values
for col in train.columns:
    print(f"Number of null values in {col} : {train.filter(column(col).isNull()).count()}")


##### Cleaning up the null values, below will be the method of clean up for each columns with null values:
- total_payment_value = 1
- total_installments = median(total_installments)
- payment_type_count = 1
- credit_card = 0 (Categorical 0 = No)
- debit_card = 0 (Categorical 0 = No)
- boleto = 0 (Categorical 0 = No)
- voucher = 0 (Categorical 0 = No)
- not_defined = 0 (Categorical 0 = No)
- total_product_weight = median(total_product_weight)
- average_product_weight = median(average_product_weight)
- max_product_weight = median(max_product_weight)
- total_product_volume = median(total_product_volume)
- min_customer_seller_distance = median(min_customer_seller_distance)
- avg_customer_seller_distance = median(avg_customer_seller_distance)
- max_customer_seller_distance = median(max_customer_seller_distance)
- min_seller_historical_late_rate = median(min_seller_historical_late_rate)
- avg_seller_historical_late_rate = median(avg_seller_historical_late_rate)
- max_seller_historical_late_rate = median(max_seller_historical_late_rate)
- primary_product_category = unknown

In [0]:
from pyspark.sql import functions as F

# Calculate medians for all columns that are going to use medians for null values
medians = train.select(
    F.median("total_installments").alias("total_installments"),
    F.median("total_product_weight").alias("total_product_weight"),
    F.median("average_product_weight").alias("average_product_weight"),
    F.median("max_product_weight").alias("max_product_weight"),
    F.median("total_product_volume").alias("total_product_volume"),
    F.median("min_customer_seller_distance").alias("min_customer_seller_distance"),
    F.median("avg_customer_seller_distance").alias("avg_customer_seller_distance"),
    F.median("max_customer_seller_distance").alias("max_customer_seller_distance"),
    F.median("min_seller_historical_late_rate").alias("min_seller_historical_late_rate"),
    F.median("avg_seller_historical_late_rate").alias("avg_seller_historical_late_rate"),
    F.median("max_seller_historical_late_rate").alias("max_seller_historical_late_rate"),
    F.median("payment_record_count").alias("payment_record_count")
).collect()[0]

# Replace null values for simpler implutations with 1s and 0s
train_features = train.fillna({
    "total_payment_value" : 1,
    "payment_type_count" : 1,
    "credit_card" : 0,
    "debit_card" : 0,
    "boleto" : 0,
    "voucher" : 0,
    "not_defined" : 0,
    "primary_product_category" : "unknown"
})

# Fill nulls with calculated medians
train = train_features.fillna({
    "total_installments": medians["total_installments"],
    "total_product_weight": medians["total_product_weight"],
    "average_product_weight": medians["average_product_weight"],
    "max_product_weight": medians["max_product_weight"],
    "total_product_volume": medians["total_product_volume"],
    "min_customer_seller_distance": medians["min_customer_seller_distance"],
    "avg_customer_seller_distance": medians["avg_customer_seller_distance"],
    "max_customer_seller_distance": medians["max_customer_seller_distance"],
    "min_seller_historical_late_rate": medians["min_seller_historical_late_rate"],
    "avg_seller_historical_late_rate": medians["avg_seller_historical_late_rate"],
    "max_seller_historical_late_rate": medians["max_seller_historical_late_rate"],
    "payment_record_count": medians["payment_record_count"]
})

# Checking for null values
for col in train.columns:
    print(f"Number of null values in {col} : {train.filter(column(col).isNull()).count()}")


In [0]:
# Apply the SAME imputation values from training to val, test, and future_data

# Validation set
val = val.fillna({
    "total_payment_value": 1,
    "payment_type_count": 1,
    "credit_card": 0,
    "debit_card": 0,
    "boleto": 0,
    "voucher": 0,
    "not_defined": 0,
    "primary_product_category": "unknown"
}).fillna({
    "total_installments": medians["total_installments"],
    "total_product_weight": medians["total_product_weight"],
    "average_product_weight": medians["average_product_weight"],
    "max_product_weight": medians["max_product_weight"],
    "total_product_volume": medians["total_product_volume"],
    "min_customer_seller_distance": medians["min_customer_seller_distance"],
    "avg_customer_seller_distance": medians["avg_customer_seller_distance"],
    "max_customer_seller_distance": medians["max_customer_seller_distance"],
    "min_seller_historical_late_rate": medians["min_seller_historical_late_rate"],
    "avg_seller_historical_late_rate": medians["avg_seller_historical_late_rate"],
    "max_seller_historical_late_rate": medians["max_seller_historical_late_rate"],
    "payment_record_count": medians["payment_record_count"]
})

# Test set
test = test.fillna({
    "total_payment_value": 1,
    "payment_type_count": 1,
    "credit_card": 0,
    "debit_card": 0,
    "boleto": 0,
    "voucher": 0,
    "not_defined": 0,
    "primary_product_category": "unknown"
}).fillna({
    "total_installments": medians["total_installments"],
    "total_product_weight": medians["total_product_weight"],
    "average_product_weight": medians["average_product_weight"],
    "max_product_weight": medians["max_product_weight"],
    "total_product_volume": medians["total_product_volume"],
    "min_customer_seller_distance": medians["min_customer_seller_distance"],
    "avg_customer_seller_distance": medians["avg_customer_seller_distance"],
    "max_customer_seller_distance": medians["max_customer_seller_distance"],
    "min_seller_historical_late_rate": medians["min_seller_historical_late_rate"],
    "avg_seller_historical_late_rate": medians["avg_seller_historical_late_rate"],
    "max_seller_historical_late_rate": medians["max_seller_historical_late_rate"],
    "payment_record_count": medians["payment_record_count"]
})

# Future data
future_data = future_data.fillna({
    "total_payment_value": 1,
    "payment_type_count": 1,
    "credit_card": 0,
    "debit_card": 0,
    "boleto": 0,
    "voucher": 0,
    "not_defined": 0,
    "primary_product_category": "unknown"
}).fillna({
    "total_installments": medians["total_installments"],
    "total_product_weight": medians["total_product_weight"],
    "average_product_weight": medians["average_product_weight"],
    "max_product_weight": medians["max_product_weight"],
    "total_product_volume": medians["total_product_volume"],
    "min_customer_seller_distance": medians["min_customer_seller_distance"],
    "avg_customer_seller_distance": medians["avg_customer_seller_distance"],
    "max_customer_seller_distance": medians["max_customer_seller_distance"],
    "min_seller_historical_late_rate": medians["min_seller_historical_late_rate"],
    "avg_seller_historical_late_rate": medians["avg_seller_historical_late_rate"],
    "max_seller_historical_late_rate": medians["max_seller_historical_late_rate"],
    "payment_record_count": medians["payment_record_count"]
})

print("Imputation applied to all datasets using training set values")

#### Encoding Categorical Columns


In [0]:
import gc

# Clear any lingering fitted model references
if 'fitted_pipeline' in dir():
    del fitted_pipeline
if 'fitted_state_indexer' in dir():
    del fitted_state_indexer
if 'fitted_state_encoder' in dir():
    del fitted_state_encoder

# Force garbage collection to free memory
gc.collect()

print("ML cache cleared")

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

# StringIndexer
# For customer state 
state_indexer = StringIndexer(
    inputCol = "customer_state",
    outputCol = "customer_state_indexed",
    # Keep unseen categories and assign them with a special index
    handleInvalid = "keep" 
)

# For primary_product_category
category_indexer = StringIndexer(
    inputCol = "primary_product_category",
    outputCol = "primary_product_category_indexed",
    handleInvalid = "keep"
)

# For target variable (late_delivery_flag)
target_indexer = StringIndexer(
    inputCol = "late_delivery_flag",
    outputCol = "late_delivery_flag_indexed",
    handleInvalid = "keep"
)

# One Hot Encoder
# For customer_state
state_encoder = OneHotEncoder(
    inputCols = ["customer_state_indexed"],
    outputCols = ["customer_state_encoded"],
    dropLast = True
)

# For primary_product_category
category_encoder = OneHotEncoder(
    inputCols = ["primary_product_category_indexed"],
    outputCols = ["primary_product_category_encoded"],
    dropLast = True
)

# Build pipeline with all encoding steps
encoding_pipeline = Pipeline(stages = [
    state_indexer,
    category_indexer,
    target_indexer,
    state_encoder,
    category_encoder
])

# Fit pipeline on training data
fitted_pipeline = encoding_pipeline.fit(train)

# Transform all datasets
train_encoded = fitted_pipeline.transform(train)
val_encoded = fitted_pipeline.transform(val)
test_encoded = fitted_pipeline.transform(test)
future_data_encoded = fitted_pipeline.transform(future_data)


# Cleaning up the original string columns
columns_to_drop = ["customer_state",
                   "customer_state_indexed",
                   "primary_product_category",
                   "primary_product_category_indexed",
                   "late_delivery_flag"]

train_final = train_encoded.drop(*columns_to_drop)
val_final = val_encoded.drop(*columns_to_drop)
test_final = test_encoded.drop(*columns_to_drop)
future_data_final = future_data_encoded.drop(*columns_to_drop)




In [0]:
# Extract label mappings from fitted StringIndexers for model interpretability

# Get the fitted stages from the pipeline
fitted_stages = fitted_pipeline.stages

# Extract customer_state mapping (first stage)
state_labels = fitted_stages[0].labels
state_mapping = {idx: label for idx, label in enumerate(state_labels)}

# Extract primary_product_category mapping (second stage)
category_labels = fitted_stages[1].labels
category_mapping = {idx: label for idx, label in enumerate(category_labels)}

# Extract late_delivery_flag mapping (third stage)
target_labels = fitted_stages[2].labels
target_mapping = {idx: label for idx, label in enumerate(target_labels)}

print("Customer State Mapping (Index -> State):")
for idx, state in state_mapping.items():
    print(f"  {idx}: {state}")

print(f"\nTotal unique states: {len(state_mapping)}")

print("\nPrimary Product Category Mapping (Index -> Category):")
for idx, category in category_mapping.items():
    print(f"  {idx}: {category}")

print(f"\nTotal unique categories: {len(category_mapping)}")

print("\nTarget Variable Mapping (Index -> Label):")
for idx, label in target_mapping.items():
    print(f"  {idx}: {label}")

# Save mappings as Python dictionaries for later use
import json

mappings = {
    "customer_state": state_mapping,
    "primary_product_category": category_mapping,
    "late_delivery_flag": target_mapping
}

print("\nMappings saved to 'mappings' variable for downstream use.")
print("Use these mappings to interpret model feature importance.")

import json
import os

# Save mappings to workspace file using standard Python I/O
workspace_path = "/Workspace/Users/prajwalparajuli2017@gmail.com/revenue_operations/artifacts"
os.makedirs(workspace_path, exist_ok=True)

mappings_file = f"{workspace_path}/encoding_mappings.json"
with open(mappings_file, "w") as f:
    json.dump(mappings, f, indent=2)

print(f"Mappings saved to: {mappings_file}")
print("\n" + "="*60)
print("ENCODING MAPPINGS REFERENCE")
print("="*60)
print("\nTarget Variable (late_delivery_flag):")
for idx, label in target_mapping.items():
    print(f"  Index {idx} = {label}")
print("\nCustomer State (first 10):")
for idx, state in list(state_mapping.items())[:10]:
    print(f"  Index {idx} = {state}")
print(f"  ... ({len(state_mapping)} total states)")
print("\nProduct Category (first 10):")
for idx, category in list(category_mapping.items())[:10]:
    print(f"  Index {idx} = {category}")
print(f"  ... ({len(category_mapping)} total categories)")

#### Encoding Documentation

**StringIndexer Transformations:**

* `customer_state`: Converts state codes (SP, RJ, MG, etc.) into numeric indices (0, 1, 2, ...). Each unique state is assigned a sequential integer. The `handleInvalid="keep"` parameter ensures that validation/test sets with previously unseen states receive a special index rather than causing errors.

* `primary_product_category`: Converts product category names into numeric indices using the same logic as customer_state. Unknown categories in validation/test data are handled gracefully.

* `late_delivery_flag`: Converts the target variable (string values) into numeric indices. This indexed version serves as the label for model training. No one-hot encoding is applied to the target variable since classification algorithms require a single numeric label column.

**OneHotEncoder Transformations:**

* `customer_state_encoded`: Transforms numeric state indices into binary vectors. For example, if there are 27 states, each state is represented by a vector of length 26 (dropLast=True removes the final category to prevent multicollinearity). A state with index 0 becomes [1, 0, 0, ...], index 1 becomes [0, 1, 0, ...], and so on.

* `primary_product_category_encoded`: Transforms numeric category indices into binary vectors using the same one-hot encoding pattern. Each category receives its own binary column in the vector.

**Pipeline Approach:**

The encoding pipeline chains StringIndexer and OneHotEncoder transformations together. The pipeline is fitted exclusively on the training set to learn the category mappings, then applied to all datasets (train, validation, test, future) to ensure consistent transformations. This prevents data leakage by ensuring that validation and test sets do not influence the learned encodings.

**Final Dataset Structure:**

After encoding and cleanup, each dataset contains:
* All original numeric features (payment values, distances, weights, etc.)
* `customer_state_encoded`: Binary vector representation of customer state
* `primary_product_category_encoded`: Binary vector representation of product category
* `late_delivery_flag_indexed`: Numeric target label (0 or 1)
* Original string columns (customer_state, primary_product_category, late_delivery_flag) and intermediate indexed columns are dropped to reduce redundancy.


In [0]:
# Save all preprocessed datasets as Delta tables for reuse
# Use overwriteSchema=True to allow schema changes (new time features added)

# Training set
train_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("revenue_operations.gold.delivery_risk_train")
print("✓ Saved training set")

# Validation set
val_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("revenue_operations.gold.delivery_risk_val")
print("✓ Saved validation set")

# Test set
test_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("revenue_operations.gold.delivery_risk_test")
print("✓ Saved test set")

# Future data
future_data_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("revenue_operations.gold.delivery_risk_future")
print("✓ Saved future data")

print("\nAll preprocessed datasets saved!")
print("\nTo use in your ML notebook:")
print("train = spark.read.table('revenue_operations.gold.delivery_risk_train')")
print("val = spark.read.table('revenue_operations.gold.delivery_risk_val')")
print("test = spark.read.table('revenue_operations.gold.delivery_risk_test')")
print("future_data = spark.read.table('revenue_operations.gold.delivery_risk_future')")